# 02 — Phase 2: Human Review UI

Review raw figure crops one patent at a time.  
**Click a thumbnail** to toggle it approved / disapproved.  
Use **Save & Advance** to confirm a patent and move to the next one.  
All decisions are written to `logs/review_meta.json` after every click (crash-safe).

**What gets shown** is controlled by `only_selected` in the cell below — no need to touch `config.yaml`.

### Controls
| Button | Effect |
|---|---|
| ← / → | Navigate between patents |
| Go / Search | Jump to a patent by ID |
| Click thumbnail | Toggle image approved / disapproved |
| Set as Main | Mark that thumbnail as the main image for this patent |
| Disapprove Patent | Reject the entire patent and advance |
| Reapprove Patent | Undo a disapproval or duplicate mark |
| Mark Duplicate | Flag patent as duplicate; enter the source patent ID |
| Needs Split | Flag the active image for sub-crop splitting (Phase 2.5) |
| 1 / 2 architectures | Toggle two-architecture mode for patents with two distinct aircraft |
| Save Arch 1 / 2 | In two-arch mode: save each selection pass separately |
| Save & Advance | Write decisions to review_meta.json and move to the next patent |

In [ ]:
import sys; sys.path.insert(0, "..")
import json
from pathlib import Path
from src.config_loader import load_config
from src.reviewer import ReviewUI

cfg = load_config()

# ── What to review (edit here — no need to touch config.yaml) ────────
only_selected = True   # True  → only patents flagged in patent_viewer.ipynb
                       # False → review every patent that was extracted (Phase 1)

if only_selected:
    sel_path = Path(cfg["paths"]["logs"]) / "selected_patents.json"
    if not sel_path.exists():
        raise FileNotFoundError(
            f"selected_patents.json not found at {sel_path}.\n"
            "Run patent_viewer.ipynb first and select the patents you want to review."
        )
    with open(sel_path) as f:
        data = json.load(f)
    patent_ids = {
        str(pid).strip().upper()
        for pid, v in data.get("patents", {}).items()
        if v.get("selected")
    }
    print(f"Reviewing {len(patent_ids)} patents selected in patent_viewer")
else:
    patent_ids = None   # show every patent folder found in crops/
    print("Reviewing ALL extracted patents")

ui = ReviewUI(cfg, patent_ids=patent_ids)
ui.show()

In [ ]:
# Run after reviewing to see totals
ui.summary()

In [ ]:
# Export review decisions to Excel (two sheets: Patents + Images)
# Output: logs/review_export.xlsx
from src.reviewer import export_review_excel
export_review_excel(cfg)